In [1]:
# general imports
import pandas as pd

# MMAI Examples

## 1. Backend setup


Both summarization APIs can run against either a local backend or a remote vLLM / OpenAI-compatible server.
The actual mode choice is shown again inside the trial and patient walkthroughs below.


### Remote backend reference

If you want either walkthrough to use a remote backend, start a server first:
```bash
    CUDA_VISIBLE_DEVICES=0 \
    python -m vllm.entrypoints.openai.api_server \
    --model openai/gpt-oss-120b \
    --download-dir ~/.cache/huggingface \
    --tensor-parallel-size 1 \
    --max-model-len 30000 \
    --gpu-memory-utilization 0.9 \
    --port 8000
```


Set environment variables for the remote backend.


In [ ]:
import os

os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"
os.environ["OPENAI_API_KEY"] = "not-needed"

### Remote config reference

You can reuse the remote config pattern shown below inside either walkthrough.


```python
from mmai import load_preset

config = load_preset("default")
config.backend = "remote"

# then pass config=config into summarize_patients(...) or summarize_trials(...)
```


### Local config reference

The walkthroughs below default to local mode unless you explicitly pass a remote config.


```python
# local mode usually just uses the default config / default backend
patient_summaries, metadata, qc_report = summarize_patients(notes, return_metadata=True, return_qc=True)
trial_results, metadata, qc_report = summarize_trials(trials, return_metadata=True, return_qc=True)
```


## 2. Trial summarization walkthrough


### Read in data

In [2]:
trials_path = "data/scheduled__2025-09-04T230000+0000.trials_for_summarize.csv"
trials = pd.read_csv(trials_path)

### Convert column names to those expected by the function.
- `trial_id`
- `trial_title`
- `brief_summary`
- `eligibility_criteria`

In [3]:
trials.rename(columns={"title": "trial_title", "nct_id": "trial_id"}, inplace=True)

### Run `summarize_trials`

`summarize_trials(...)` supports both modes:
- Local mode: call it directly with the default config
- Remote mode: create a preset with `config.backend = "remote"` and pass `config=config`

The next cell shows the default local example.


In [4]:
# import and run function on trial data
from mmai.trials import summarize_trials

trial_results, metadata, qc_report = summarize_trials(
    trials, return_metadata=True, return_qc=True
)

/home/sabrina/mmai-package/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 04-13 21:17:01 [utils.py:233] non-default args: {'max_model_len': 10000, 'disable_log_stats': True, 'model': 'openai/gpt-oss-120b'}
INFO 04-13 21:17:02 [model.py:533] Resolved architecture: GptOssForCausalLM


Parse safetensors files: 100%|██████████| 15/15 [00:00<00:00, 70.27it/s]

INFO 04-13 21:17:02 [model.py:1582] Using max model len 10000


INFO 04-13 21:17:04 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-13 21:17:04 [config.py:78] Overriding max cuda graph capture size to 1024 for performance.
INFO 04-13 21:17:04 [vllm.py:775] Asynchronous scheduling is enabled.
(EngineCore pid=133647) INFO 04-13 21:17:08 [core.py:103] Initializing a V1 LLM engine (v0.18.1) with config: model='openai/gpt-oss-120b', speculative_config=None, tokenizer='openai/gpt-oss-120b', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=10000, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=mxfp4, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto'

(EngineCore pid=133647) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=133647) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/15 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   7% Completed | 1/15 [00:01<00:18,  1.29s/it]
Loading safetensors checkpoint shards:  13% Completed | 2/15 [00:02<00:16,  1.25s/it]
Loading safetensors checkpoint shards:  20% Completed | 3/15 [00:03<00:15,  1.28s/it]
Loading safetensors checkpoint shards:  27% Completed | 4/15 [00:05<00:13,  1.25s/it]
Loading safetensors checkpoint shards:  33% Completed | 5/15 [00:06<00:12,  1.28s/it]
Loading safetensors checkpoint shards:  40% C

(EngineCore pid=133647) INFO 04-13 21:17:30 [default_loader.py:384] Loading weights took 18.88 seconds
(EngineCore pid=133647) INFO 04-13 21:17:34 [gpu_model_runner.py:4566] Model loading took 65.97 GiB memory and 23.598404 seconds
(EngineCore pid=133647) INFO 04-13 21:17:46 [backends.py:988] Using cache directory: /home/sabrina/.cache/vllm/torch_compile_cache/7ce6c67300/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=133647) INFO 04-13 21:17:46 [backends.py:1048] Dynamo bytecode transform time: 11.05 s
(EngineCore pid=133647) INFO 04-13 21:17:47 [backends.py:371] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=133647) INFO 04-13 21:17:49 [backends.py:387] Compiling a graph for compile range (1, 8192) takes 2.09 s
(EngineCore pid=133647) INFO 04-13 21:17:51 [decorators.py:627] saved AOT compiled function to /home/sabrina/.cache/vllm/torch_compile_cache/torch_aot_compile/c557e614c87041e4a395ca2d888c361cda916b84c3d136d4f50ae444d9988f68/rank_0_0/model
(

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 83/83 [00:10<00:00,  8.24it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:03<00:00,  8.87it/s]


(EngineCore pid=133647) INFO 04-13 21:18:10 [gpu_model_runner.py:5746] Graph capturing finished in 15 secs, took 1.08 GiB
(EngineCore pid=133647) INFO 04-13 21:18:10 [gpu_worker.py:617] CUDA graph pool memory: 1.08 GiB (actual), 0.96 GiB (estimated), difference: 0.12 GiB (11.0%).
(EngineCore pid=133647) INFO 04-13 21:18:10 [core.py:281] init engine (profile, create kv cache, warmup model) took 36.37 seconds
INFO 04-13 21:18:14 [llm.py:391] Supported tasks: ['generate']


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4350.52it/s]


### View results

In [5]:
trial_results.head()

,trial_id,clinical_space_number,clinical_space_summary,general_exclusion_criteria,space_trial_id
0,NCT03319901,1,Age range allowed: ≥60 years. Sex allowed: Any...,Pregnancy or breastfeeding; uncontrolled infec...,NCT03319901-1
1,NCT03319901,2,Age range allowed: ≥60 years. Sex allowed: Any...,Pregnancy or breastfeeding; uncontrolled infec...,NCT03319901-2
2,NCT03319901,3,Age range allowed: ≥18 years. Sex allowed: Any...,Pregnancy or breastfeeding; uncontrolled infec...,NCT03319901-3
3,NCT03319901,4,Age range allowed: ≥18 years. Sex allowed: Any...,Pregnancy or breastfeeding; uncontrolled infec...,NCT03319901-4
4,NCT04792489,1,Age range allowed: ≥18 years. Sex allowed: bot...,History of infection with hiv or active hbv su...,NCT04792489-1


Metadata contains a snapshot of the config used and metadata on the model used. 

In [6]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'backend': 'local',
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'max_model_len': 10000,
   'tensor_parallel_size': 1,
   'gpu_memory_utilization': 0.9,
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'max_model_len': 100000,
   'tensor_parallel_size': 1,
   'gpu_memory_utilization': 0.9,
   'chunk_size': 10000,
   'chunk_overlap': 500,
   'prompt_margin_tokens': 5000,
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.2},
   'prompt_files': {'primer': 'patient.serial.user.primer.txt',
    'quest

QC report on the trial summarization run

In [7]:
qc_report

,metric,value,denominator,percent,ids
0,trials_missing_in_output,2.0,470.0,0.425532,"[NCT05653882, NCT05941507]"
1,trials_truncated_llm_response,1.0,470.0,0.212766,[NCT05417594]
2,spaces_exceed_embedding_token_limit,0.0,1460.0,0.000000,[]
3,spaces_per_trial_min,1.0,NaN,NaN,[]
4,spaces_per_trial_median,2.0,NaN,NaN,[]
5,spaces_per_trial_max,33.0,NaN,NaN,[]
6,trials_with_non_distinct_spaces,4.0,468.0,0.854701,"[NCT04777994, NCT05137054, NCT05259839, NCT055..."
7,spaces_dropped_missing_keyword:Age,0.0,1464.0,0.000000,[]
8,spaces_dropped_missing_keyword:Sex,0.0,1464.0,0.000000,[]
9,spaces_dropped_missing_keyword:Cancer type,0.0,1464.0,0.000000,[]


Save trial example results


In [8]:
import json

trial_results.to_csv("output/trial_summaries.csv", index=None)
with open("output/trial_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/trial_qc_report.csv", index=None)

## 3. Patient summarization walkthrough


### Read in data

In [2]:
notes_path = "data/2026_01_16_dummy_notes.csv"
notes = pd.read_csv(notes_path)
notes.head()

,DFCI_MRN,EVENT_DATE,RPT_TEXT,NOTE_SOURCE,LAST_MOD,PROVIDER_TYPE,RPT_STATUS,ENCOUNTER_TYPE_DESC,PROVIDER_DEPT,PROC_DESC
0,-1,6/20/25,**Imaging Report**\n\n---\n\n**Patient:** Mich...,Imaging,6/25/25,PROVIDER_TYPE_NM,NOTE_STATUS_DESCR,ENCOUNTER_TYPE_DESCR,NaN,NaN
1,-1,6/25/25,**Oncology Progress Note – Follow‑Up Visit**\n...,progress_notesbase,6/25/25,Physician,Signed,Office Visit,NaN,NaN
2,-2,6/19/25,**Imaging Report**\n\n**Patient:** Daniel O’Co...,Imaging,6/25/25,PROVIDER_TYPE_NM,NOTE_STATUS_DESCR,ENCOUNTER_TYPE_DESCR,NaN,NaN
3,-2,6/20/25,**Oncology Progress Note**\n\n**Patient:** Dan...,progress_notesbase,6/25/25,Physician,Signed,Office Visit,NaN,NaN
4,-2,6/25/25,**Oncology Progress Note – Follow‑up Visit**\n...,progress_notesbase,6/25/25,Physician,Signed,Office Visit,NaN,NaN


### Convert column names to those expected by the function.
- `patient_id`
- `note_text`
- `note_date`

In [3]:
notes.rename(
    columns={
        "DFCI_MRN": "patient_id",
        "RPT_TEXT": "note_text",
        "EVENT_DATE": "note_date",
    },
    inplace=True,
)

### Run `summarize_patients`

`summarize_patients(...)` supports both modes:
- Local mode: call it directly with the default config
- Remote mode: create a preset with `config.backend = "remote"` and pass `config=config`

The next cell shows the default local example.


In [4]:
from mmai.patients import summarize_patients

patient_summaries, metadata, qc_report = summarize_patients(
    notes, return_metadata=True, return_qc=True
)

/home/sabrina/mmai-package/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/sabrina/mmai-package/src/mmai/patients/prepare.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  normalized["note_date"] = pd.to_datetime(normalized["note_date"])


INFO 04-13 21:34:54 [utils.py:233] non-default args: {'max_model_len': 100000, 'disable_log_stats': True, 'model': 'openai/gpt-oss-120b'}
INFO 04-13 21:34:55 [model.py:533] Resolved architecture: GptOssForCausalLM


Parse safetensors files: 100%|██████████| 15/15 [00:00<00:00, 72.82it/s]

INFO 04-13 21:34:55 [model.py:1582] Using max model len 100000


INFO 04-13 21:34:56 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-13 21:34:56 [config.py:78] Overriding max cuda graph capture size to 1024 for performance.
INFO 04-13 21:34:56 [vllm.py:775] Asynchronous scheduling is enabled.
(EngineCore pid=140423) INFO 04-13 21:35:00 [core.py:103] Initializing a V1 LLM engine (v0.18.1) with config: model='openai/gpt-oss-120b', speculative_config=None, tokenizer='openai/gpt-oss-120b', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=100000, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=mxfp4, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto

(EngineCore pid=140423) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=140423) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/15 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   7% Completed | 1/15 [00:01<00:17,  1.28s/it]
Loading safetensors checkpoint shards:  13% Completed | 2/15 [00:02<00:15,  1.22s/it]
Loading safetensors checkpoint shards:  20% Completed | 3/15 [00:03<00:15,  1.26s/it]
Loading safetensors checkpoint shards:  27% Completed | 4/15 [00:04<00:13,  1.23s/it]
Loading safetensors checkpoint shards:  33% Completed | 5/15 [00:06<00:12,  1.26s/it]
Loading safetensors checkpoint shards:  40% C

(EngineCore pid=140423) INFO 04-13 21:35:22 [default_loader.py:384] Loading weights took 18.62 seconds
(EngineCore pid=140423) INFO 04-13 21:35:26 [gpu_model_runner.py:4566] Model loading took 65.97 GiB memory and 23.304253 seconds
(EngineCore pid=140423) INFO 04-13 21:35:38 [backends.py:988] Using cache directory: /home/sabrina/.cache/vllm/torch_compile_cache/69fec100f0/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=140423) INFO 04-13 21:35:38 [backends.py:1048] Dynamo bytecode transform time: 11.15 s
(EngineCore pid=140423) INFO 04-13 21:35:40 [backends.py:371] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=140423) INFO 04-13 21:35:44 [backends.py:387] Compiling a graph for compile range (1, 8192) takes 5.30 s
(EngineCore pid=140423) INFO 04-13 21:35:47 [decorators.py:627] saved AOT compiled function to /home/sabrina/.cache/vllm/torch_compile_cache/torch_aot_compile/92b975c5678a9438bf87f0fee4a1a4be4eed54b96459a201a7cd607ba8919fd0/rank_0_0/model
(

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 83/83 [00:10<00:00,  8.11it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:04<00:00,  8.57it/s]


(EngineCore pid=140423) INFO 04-13 21:36:07 [gpu_model_runner.py:5746] Graph capturing finished in 15 secs, took 1.08 GiB
(EngineCore pid=140423) INFO 04-13 21:36:07 [gpu_worker.py:617] CUDA graph pool memory: 1.08 GiB (actual), 0.96 GiB (estimated), difference: 0.12 GiB (11.0%).
(EngineCore pid=140423) INFO 04-13 21:36:07 [core.py:281] init engine (profile, create kv cache, warmup model) took 40.61 seconds
INFO 04-13 21:36:11 [llm.py:391] Supported tasks: ['generate']


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4208.89it/s]


### View results

In [5]:
patient_summaries

,patient_id,last_note_date,cancer_history_summary,general_exclusion_criteria_evidence
0,-1,2025-06-25,Age: 64\nSex: Male\nCancer type: Gastroesophag...,No evidence of common boilerplate exclusion cr...
1,-10,2025-06-25,Age: 55 \nSex: Female \nCancer type: Ovarian...,No evidence of common boilerplate exclusion cr...
2,-11,2025-06-25,Age: 68 \nSex: Male \nCancer type: Cutaneous...,Prior immune‑related adverse events include Gr...
3,-12,2025-06-25,Age: 19 \nSex: Male \nCancer type: Osteosarc...,No evidence of common exclusion criteria – no ...
4,-2,2025-06-25,Age: 34\nSex: Male\nCancer type: Appendix neur...,No evidence of common boilerplate exclusion cr...
5,-3,2025-06-25,Age: 66\nSex: Male\nCancer type: Breast cancer...,No evidence of common boilerplate exclusion cr...
6,-4,2025-06-25,Age: 62 \nSex: Female \nCancer type: Gliobla...,Presence of hypertension (BP recordings around...
7,-5,2025-06-25,Age: 45\nSex: Female\nCancer type: Breast canc...,No evidence of common boilerplate exclusion cr...
8,-6,2025-06-25,Age: 68\nSex: Male\nCancer type: Small‑cell lu...,No evidence of common exclusion criteria such ...
9,-7,2025-06-25,Age: 70\nSex: Male\nCancer type: Prostate canc...,Controlled essential hypertension on amlodipin...


Metadata contains a snapshot of the config used and metadata on the models used. 

In [6]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'backend': 'local',
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'max_model_len': 10000,
   'tensor_parallel_size': 1,
   'gpu_memory_utilization': 0.9,
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'max_model_len': 100000,
   'tensor_parallel_size': 1,
   'gpu_memory_utilization': 0.9,
   'chunk_size': 10000,
   'chunk_overlap': 500,
   'prompt_margin_tokens': 5000,
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.2},
   'prompt_files': {'primer': 'patient.serial.user.primer.txt',
    'quest

QC report on patient summarization

In [7]:
qc_report

,metric,value,denominator,percent,ids
0,patients_dropped_noninformative_summary,0,12,0.0,[]
1,patients_exceed_embedding_token_limit,0,12,0.0,[]
2,patients_exclusion_criteria_not_extracted,0,12,0.0,[]
3,patients_missing_keyword:Age,0,12,0.0,[]
4,patients_missing_keyword:Sex,0,12,0.0,[]
5,patients_missing_keyword:Cancer type,0,12,0.0,[]
6,patients_missing_keyword:Histology,0,12,0.0,[]
7,patients_missing_keyword:Current extent,0,12,0.0,[]
8,patients_missing_keyword:Biomarkers,0,12,0.0,[]
9,patients_missing_keyword:Treatment history,0,12,0.0,[]


Save example results

In [9]:
patient_summaries.to_csv("output/patient_summaries.csv", index=None)
with open("output/patient_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/patient_qc_report.csv", index=None)

## 4. Embed summarized entities for matching


In [2]:
from mmai.embedding import embed_for_matching

In [ ]:
# Uncomment if you want to start from previously generated summaries
# trial_results = pd.read_csv("output/trial_summaries.csv")
# patient_summaries = pd.read_csv("output/patient_summaries.csv", dtype="str")

### Trials

In [ ]:
embedded_trials, metadata = embed_for_matching(
    trial_results, entity_type="trial", return_metadata=True
)

/home/sabrina/mmai-package/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4278.79it/s]


In [5]:
embedded_trials.head()

,space_trial_id,embedding
0,NCT03319901-1,"[0.04174385964870453, 0.01991516910493374, -0...."
1,NCT03319901-2,"[0.020876429975032806, 0.041405949741601944, -..."
2,NCT03319901-3,"[0.04114490747451782, 0.011669274419546127, -0..."
3,NCT03319901-4,"[0.014145676977932453, 0.04072318226099014, 0...."
4,NCT04792489-1,"[-0.008480580523610115, -0.043102700263261795,..."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [6]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'backend': 'local',
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'max_model_len': 10000,
   'tensor_parallel_size': 1,
   'gpu_memory_utilization': 0.9,
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'max_model_len': 100000,
   'tensor_parallel_size': 1,
   'gpu_memory_utilization': 0.9,
   'chunk_size': 10000,
   'chunk_overlap': 500,
   'prompt_margin_tokens': 5000,
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.2},
   'prompt_files': {'primer': 'patient.serial.user.primer.txt',
    'quest

### Patients

In [ ]:
embedded_patients, metadata = embed_for_matching(
    patient_summaries, entity_type="patient", return_metadata=True
)

In [8]:
embedded_patients.head()

,patient_id,embedding
0,-1,"[0.05117025971412659, -0.014803430996835232, -..."
1,-10,"[-0.056036848574876785, -0.033462148159742355,..."
2,-11,"[0.032251086086034775, -4.447812898433767e-05,..."
3,-12,"[0.0020424681715667248, -0.018599383533000946,..."
4,-2,"[-0.0004920277860946953, -0.020355189219117165..."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [9]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'backend': 'local',
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'max_model_len': 10000,
   'tensor_parallel_size': 1,
   'gpu_memory_utilization': 0.9,
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'max_model_len': 100000,
   'tensor_parallel_size': 1,
   'gpu_memory_utilization': 0.9,
   'chunk_size': 10000,
   'chunk_overlap': 500,
   'prompt_margin_tokens': 5000,
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.2},
   'prompt_files': {'primer': 'patient.serial.user.primer.txt',
    'quest

In [ ]:
embedded_patients.to_parquet("output/embedded_patients.parquet")
embedded_trials.to_parquet("output/embedded_trials.parquet")

## 5. Generate candidate matches in both directions

`generate_candidate_matches(...)` is symmetric: whichever dataframe is passed as `query_df` becomes the perspective used for ranking.


In [12]:
from mmai.matching import generate_candidate_matches

In [ ]:
# Uncomment if you want to start from previously generated embeddings
# embedded_patients = pd.read_parquet('output/embedded_patients.parquet')
# embedded_trials = pd.read_parquet('output/embedded_trials.parquet')

### Patient-centric matching

Patients are the queries and trial spaces are the corpus.


In [13]:
patient_centric_candidate_matches = generate_candidate_matches(
    query_df=embedded_patients, corpus_df=embedded_trials
)

In [14]:
patient_centric_candidate_matches.head()

,patient_id,space_trial_id,similarity_score,rank
0,-1,NCT05372614-2,0.588519,1
1,-1,NCT05482893-6,0.559669,2
2,-1,NCT06244485-2,0.547970,3
3,-1,NCT06293898-8,0.546890,4
4,-1,NCT04389632-14,0.546686,5


### Trial-centric matching

Trial spaces are the queries and patients are the corpus.


In [15]:
trial_centric_candidate_matches = generate_candidate_matches(
    query_df=embedded_trials, corpus_df=embedded_patients
)

In [16]:
trial_centric_candidate_matches.head()

,space_trial_id,patient_id,similarity_score,rank
0,NCT03319901-1,-7,0.104453,1
1,NCT03319901-1,-5,0.091949,2
2,NCT03319901-1,-6,0.036489,3
3,NCT03319901-1,-8,0.027320,4
4,NCT03319901-1,-3,0.021978,5


## 6. Patient-centric reasonable match check

The reasonable-match and exclusion-criteria checkers are currently set up for patient-centric candidate matches only.


In [17]:
from mmai.matching import reasonable_match_check

Add back the patient and trial summary context required by the patient-centric reasonable-match checker.


In [ ]:
patient_centric_candidate_matches_with_context = (
    patient_centric_candidate_matches.merge(
        patient_summaries[["patient_id", "cancer_history_summary"]],
        on="patient_id",
        how="left",
    ).merge(
        trial_results[["space_trial_id", "clinical_space_summary"]],
        on="space_trial_id",
        how="left",
    )
)
patient_centric_candidate_matches_with_context.head()

,patient_id,space_trial_id,similarity_score,rank,cancer_history_summary,clinical_space_summary
0,-1,NCT05372614-2,0.588519,1,Age: 64\nSex: Male\nCancer type: Gastroesophag...,Age range allowed: ≥ 18 years. Sex allowed: Bo...
1,-1,NCT05482893-6,0.559669,2,Age: 64\nSex: Male\nCancer type: Gastroesophag...,Age range allowed: ≥ 18 years. Sex allowed: Bo...
2,-1,NCT06244485-2,0.547970,3,Age: 64\nSex: Male\nCancer type: Gastroesophag...,Age range allowed: ≥ 18 years. Sex allowed: ma...
3,-1,NCT06293898-8,0.546890,4,Age: 64\nSex: Male\nCancer type: Gastroesophag...,Age range allowed: ≥18 years. Sex allowed: Mal...
4,-1,NCT04389632-14,0.546686,5,Age: 64\nSex: Male\nCancer type: Gastroesophag...,Age range allowed: NA. Sex allowed: Male and F...


Run the patient-centric reasonable-match checker.


In [ ]:
patient_centric_reasonable_matches, metadata = reasonable_match_check(
    patient_centric_candidate_matches_with_context, return_metadata=True
)

Loading weights: 100%|██████████| 174/174 [00:00<00:00, 3867.62it/s]


In [20]:
patient_centric_reasonable_matches.head()

,patient_id,space_trial_id,reasonable_match_score,reasonable_match
0,-1,NCT05372614-2,0.988166,True
1,-1,NCT05482893-6,0.594984,True
2,-1,NCT06244485-2,0.997060,True
3,-1,NCT06293898-8,0.984553,True
4,-1,NCT04389632-14,0.780814,True


In [21]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'backend': 'local',
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'max_model_len': 10000,
   'tensor_parallel_size': 1,
   'gpu_memory_utilization': 0.9,
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'max_model_len': 100000,
   'tensor_parallel_size': 1,
   'gpu_memory_utilization': 0.9,
   'chunk_size': 10000,
   'chunk_overlap': 500,
   'prompt_margin_tokens': 5000,
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.2},
   'prompt_files': {'primer': 'patient.serial.user.primer.txt',
    'quest

## 7. Patient-centric exclusion criteria check


In [ ]:
# limit to unique patient-trial ID pairs
patient_centric_reasonable_matches["trial_id"] = (
    patient_centric_reasonable_matches["space_trial_id"].str.split("-").str[0]
)
unique_patient_trial_pairs = patient_centric_reasonable_matches[
    ["patient_id", "trial_id"]
].drop_duplicates()

# add in exclusion criteria for patients and trials
patient_trial_pairs_with_exclusion_context = unique_patient_trial_pairs.merge(
    patient_summaries[["patient_id", "general_exclusion_criteria_evidence"]],
    on="patient_id",
    how="left",
).merge(
    trial_results[["trial_id", "general_exclusion_criteria"]].drop_duplicates(),
    on="trial_id",
    how="left",
)
patient_trial_pairs_with_exclusion_context.head()

,patient_id,trial_id,general_exclusion_criteria_evidence,general_exclusion_criteria
0,-1,NCT05372614,No evidence of common boilerplate exclusion cr...,Patients must not receive other anticancer the...
1,-1,NCT05482893,No evidence of common boilerplate exclusion cr...,Women who are pregnant or lactating.; Women of...
2,-1,NCT06244485,No evidence of common boilerplate exclusion cr...,Has uncontrolled or significant cardiovascular...
3,-1,NCT06293898,No evidence of common boilerplate exclusion cr...,History of severe heart disease; left ventricu...
4,-1,NCT04389632,No evidence of common boilerplate exclusion cr...,History of another malignancy within 3 years p...


In [ ]:
from mmai.matching import exclusion_criteria_check

# run exclusion criteria check
exclusion_results, metadata = exclusion_criteria_check(
    patient_trial_pairs_with_exclusion_context, return_metadata=True
)
exclusion_results.head()

Loading weights: 100%|██████████| 174/174 [00:00<00:00, 3331.82it/s]


,patient_id,trial_id,exclusion_score,exclusion_criteria_pass
0,-1,NCT05372614,0.999986,True
1,-1,NCT05482893,0.999995,True
2,-1,NCT06244485,0.999996,True
3,-1,NCT06293898,0.999986,True
4,-1,NCT04389632,0.999993,True


In [ ]:
exclusion_results["exclusion_criteria_pass"].value_counts()

exclusion_criteria_pass
True     149
False      6
Name: count, dtype: int64

In [25]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'backend': 'local',
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'max_model_len': 10000,
   'tensor_parallel_size': 1,
   'gpu_memory_utilization': 0.9,
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'max_model_len': 100000,
   'tensor_parallel_size': 1,
   'gpu_memory_utilization': 0.9,
   'chunk_size': 10000,
   'chunk_overlap': 500,
   'prompt_margin_tokens': 5000,
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.2},
   'prompt_files': {'primer': 'patient.serial.user.primer.txt',
    'quest